# Nhận diện biển báo giao thông bằng mạng CNN (GTSRB + biển báo Việt Nam)

Mô hình CNN huấn luyện trên 43 loại biển của GTSRB và 22 loại biển chỉ có ở Việt Nam (cấm dừng đỗ, cấm đỗ, cấm rẽ, cấm quay đầu, tốc độ 40 km/h...). Biển Việt Nam giống biển GTSRB được gộp vào nhãn cũ.


In [ ]:
# Bước 1: Cài thư viện và kiểm tra GPU
!pip install -q kagglehub
import os, glob, time, json, random, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
import tensorflow as tf
import kagglehub
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Bước 2: Tải hai bộ dữ liệu
# GTSRB: 43 loại biển (Đức), ảnh đã cắt sẵn
# Vietnamese Traffic Signs: ảnh chụp đường phố Việt Nam, biển được khoanh vùng (định dạng YOLO)
duong_dan = kagglehub.dataset_download('meowmeowmeowmeowmeow/gtsrb-german-traffic-sign')
ROOT = None
for d, thu_muc_con, cac_file in os.walk(duong_dan):
    if 'Train' in thu_muc_con and 'Test.csv' in cac_file:
        ROOT = d
        break
vn = kagglehub.dataset_download('maitam/vietnamese-traffic-signs')
VN = os.path.dirname(glob.glob(vn + '/**/classes.txt', recursive=True)[0])
print('GTSRB:', ROOT)
print('Biển Việt Nam:', VN)

In [ ]:
# Bước 3: Đọc ảnh GTSRB (giống traffic_sign.py)
data = []
labels = []
for i in range(43):
    path = os.path.join(ROOT, 'Train', str(i))
    for a in os.listdir(path):
        try:
            image = Image.open(os.path.join(path, a)).convert('RGB')
            image = image.resize((30, 30))
            data.append(np.array(image))
            labels.append(i)
        except Exception:
            print('Lỗi đọc ảnh:', a)
data = np.array(data)
labels = np.array(labels)
print('GTSRB:', data.shape)

In [ ]:
# Bước 4: Đọc biển báo Việt Nam
# Biển giống biển GTSRB thì gộp vào nhãn cũ (bảng GHEP).
# Biển chưa có thì thêm thành nhãn mới 43, 44, ... (danh sách THEM).
ten_lop = [l.strip() for l in open(os.path.join(VN, 'classes.txt'), encoding='utf-8') if l.strip()]
ten_vie = [l.strip() for l in open(os.path.join(VN, 'classes_vie.txt'), encoding='utf-8') if l.strip()]
TEN_VIE = dict(zip(ten_lop, ten_vie)) if len(ten_lop) == len(ten_vie) else {}

GHEP = {'P.102': 17, 'P.127*50': 2, 'P.127*60': 3, 'P.127*80': 5, 'P.106a*Xe tải': 16, 'R.302a': 38, 'R.303': 40,
        'W.201a': 19, 'W.201b': 20, 'W.203c': 24, 'W.207': 11, 'W.221b': 22, 'W.224': 27, 'W.225': 28, 'W.233': 18}
THEM = ['P.130', 'P.131a', 'P.123a', 'P.123b', 'P.124a*', 'P.124c', 'P.127*40', 'P.103a', 'P.104', 'P.107', 'P.111',
        'P.117*', 'P.129', 'P.137', 'P.139', 'W.245a', 'W.205b', 'W.210', 'W.246a', 'I.407a', 'I.434a', 'R.415a']
NHAN = dict(GHEP)
for k, t in enumerate(THEM):
    NHAN[t] = 43 + k
SO_LOP = 43 + len(THEM)

# Bộ dữ liệu đã chia sẵn ảnh dùng để huấn luyện và ảnh dùng để kiểm tra
def doc_ds(ten):
    f = glob.glob(VN + '/**/' + ten, recursive=True)
    return set(os.path.splitext(os.path.basename(l.strip()))[0] for l in open(f[0])) if f else set()
anh_train = doc_ds('train_files.txt')
anh_test = doc_ds('test_files.txt')

vn_X, vn_y, vn_phan, vn_ten, vn_hop = [], [], [], [], []
for f in sorted(glob.glob(os.path.join(VN, 'labels', '*.txt'))):
    ma_anh = os.path.basename(f)[:-4]
    anh = os.path.join(VN, 'images', ma_anh + '.jpg')
    if not os.path.exists(anh):
        continue
    im = Image.open(anh).convert('RGB')
    W, H = im.size
    for dong in open(f):
        v = dong.split()
        if len(v) < 5 or ten_lop[int(v[0])] not in NHAN:
            continue
        cx, cy, w, h = float(v[1]) * W, float(v[2]) * H, float(v[3]) * W, float(v[4]) * H
        bien = im.crop((max(0, cx - 0.6 * w), max(0, cy - 0.6 * h), min(W, cx + 0.6 * w), min(H, cy + 0.6 * h)))
        if min(bien.size) < 8:
            continue
        vn_X.append(np.array(bien.resize((30, 30))))
        vn_y.append(NHAN[ten_lop[int(v[0])]])
        vn_phan.append('test' if ma_anh in anh_test else 'train')
        vn_ten.append(ten_lop[int(v[0])])
        vn_hop.append((anh, cx, cy, w, h))
vn_X, vn_y, vn_phan = np.array(vn_X), np.array(vn_y), np.array(vn_phan)
print('Biển Việt Nam: huấn luyện', (vn_phan == 'train').sum(), '| kiểm tra', (vn_phan == 'test').sum(), '| tổng số lớp:', SO_LOP)
print(pd.Series(vn_ten).value_counts().to_string())

In [ ]:
# Bước 5: Ghép dữ liệu
# Ảnh Việt Nam ít hơn GTSRB nhiều nên mỗi ảnh được dùng 5 lần, mỗi lần xoay và đổi độ sáng một chút.
def bien_doi(a):
    im = Image.fromarray(a).rotate(random.uniform(-10, 10), resample=Image.BILINEAR)
    im = ImageEnhance.Brightness(im).enhance(random.uniform(0.6, 1.4))
    return np.array(ImageEnhance.Contrast(im).enhance(random.uniform(0.8, 1.2)))

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
random.seed(42)
X_train, X_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, random_state=42)
vn_tr_X, vn_tr_y = vn_X[vn_phan == 'train'], vn_y[vn_phan == 'train']
vn_te_X, vn_te_y = vn_X[vn_phan == 'test'], vn_y[vn_phan == 'test']
vt_X, vv_X, vt_y, vv_y = train_test_split(vn_tr_X, vn_tr_y, test_size=0.15, random_state=42)
them_X = [vt_X] + [np.array([bien_doi(a) for a in vt_X]) for _ in range(4)]
X_train = np.concatenate([X_train] + them_X)
y_train = np.concatenate([y_train] + [vt_y] * 5)
X_val = np.concatenate([X_val, vv_X])
y_val = np.concatenate([y_val, vv_y])
print('Huấn luyện:', X_train.shape, '| kiểm định:', X_val.shape)

In [ ]:
# Bước 6: Mô hình CNN (kiến trúc giữ nguyên như traffic_sign.py, lớp cuối có SO_LOP đầu ra)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout
model = Sequential()
model.add(Conv2D(filters=32, kernel_size=(5, 5), padding='same', activation='relu', input_shape=(30, 30, 3)))
model.add(Conv2D(filters=32, kernel_size=(5, 5), padding='same', activation='relu'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(rate=0.25))
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(rate=0.25))
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(rate=0.5))
model.add(Dense(SO_LOP, activation='softmax'))
model.summary()

epochs = 20
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
bat_dau = time.time()
history = model.fit(X_train, to_categorical(y_train, SO_LOP), batch_size=32, epochs=epochs,
                    validation_data=(X_val, to_categorical(y_val, SO_LOP)))
thoi_gian_huan_luyen = (time.time() - bat_dau) / 60
model.save('traffic_classifier.h5')

In [ ]:
# Bước 7: Đồ thị độ chính xác và hàm mất mát
ep = range(1, epochs + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, history.history['accuracy'], 'o-', label='Huấn luyện')
ax[0].plot(ep, history.history['val_accuracy'], 's-', label='Kiểm định')
ax[0].set_title('Độ chính xác')
ax[0].set_xlabel('Epoch')
ax[0].legend()
ax[0].grid(alpha=0.3)
ax[1].plot(ep, history.history['loss'], 'o-', label='Huấn luyện')
ax[1].plot(ep, history.history['val_loss'], 's-', label='Kiểm định')
ax[1].set_title('Hàm mất mát')
ax[1].set_xlabel('Epoch')
ax[1].legend()
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('do_thi_huan_luyen.png', dpi=200)
plt.show()

In [ ]:
# Bước 8: Kiểm tra
# (a) 8.000 ảnh Test của GTSRB; (b) toàn bộ biển trong các ảnh kiểm tra của bộ Việt Nam (không dùng khi huấn luyện)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
test = pd.read_csv(os.path.join(ROOT, 'Test.csv'))
test = test.sample(min(8000, len(test)), random_state=42)
X8 = np.array([np.array(Image.open(os.path.join(ROOT, p)).convert('RGB').resize((30, 30))) for p in test['Path']])
y8 = test['ClassId'].values
p8 = model.predict(X8, verbose=0).argmax(axis=1)
P, R, F1, so_anh = precision_recall_fscore_support(y8, p8, labels=list(range(43)), zero_division=0)
pd.DataFrame({'Nhãn': range(43), 'Số ảnh': so_anh, 'Precision': P.round(2), 'Recall': R.round(2), 'F1': F1.round(2)}).to_csv('f1_gtsrb.csv', index=False)

pv = model.predict(vn_te_X, verbose=0).argmax(axis=1)
nhan_vn = sorted(set(vn_te_y))
Pv, Rv, Fv, Sv = precision_recall_fscore_support(vn_te_y, pv, labels=nhan_vn, zero_division=0)
TEN_NHAN = {n: t for t, n in NHAN.items()}
bang_vn = pd.DataFrame({'Nhãn': nhan_vn, 'Biển': [TEN_NHAN.get(n, '') for n in nhan_vn], 'Số biển': Sv,
                        'Precision': Pv.round(2), 'Recall': Rv.round(2), 'F1': Fv.round(2)})
bang_vn.to_csv('f1_bien_viet_nam.csv', index=False, encoding='utf-8-sig')
print(bang_vn.to_string(index=False))
moi = vn_te_y >= 43
print('GTSRB 8.000 ảnh:', round(accuracy_score(y8, p8) * 100, 2), '%')
print('Biển Việt Nam:', round(accuracy_score(vn_te_y, pv) * 100, 2), '% | biển cũ', round(accuracy_score(vn_te_y[~moi], pv[~moi]) * 100, 2),
      '% | biển mới', round(accuracy_score(vn_te_y[moi], pv[moi]) * 100, 2), '%')

In [ ]:
# Bước 9: Chuyển sang TensorFlow Lite cho app (batch = 1, float32)
m1 = Sequential()
m1.add(tf.keras.layers.Input((30, 30, 3), batch_size=1))
for layer in model.layers:
    m1.add(layer.__class__.from_config(layer.get_config()))
m1.set_weights(model.get_weights())
open('model.tflite', 'wb').write(tf.lite.TFLiteConverter.from_keras_model(m1).convert())
it = tf.lite.Interpreter(model_path='model.tflite')
it.allocate_tensors()
VAO = it.get_input_details()[0]['index']
RA = it.get_output_details()[0]['index']
print('model.tflite:', round(os.path.getsize('model.tflite') / 2**20, 2), 'MB, đầu ra', it.get_output_details()[0]['shape'])

In [ ]:
# Bước 10: Tên biển cho app (signs.json) và ảnh minh họa cho biển mới
SIGNS_43 = json.loads('''[["P.127", "Tốc độ tối đa 20 km/h", "Biển báo cấm", "Cấm các loại xe cơ giới chạy với tốc độ vượt quá 20 km/h."], ["P.127", "Tốc độ tối đa 30 km/h", "Biển báo cấm", "Cấm các loại xe cơ giới chạy với tốc độ vượt quá 30 km/h."], ["P.127", "Tốc độ tối đa 50 km/h", "Biển báo cấm", "Cấm các loại xe cơ giới chạy với tốc độ vượt quá 50 km/h."], ["P.127", "Tốc độ tối đa 60 km/h", "Biển báo cấm", "Cấm các loại xe cơ giới chạy với tốc độ vượt quá 60 km/h."], ["P.127", "Tốc độ tối đa 70 km/h", "Biển báo cấm", "Cấm các loại xe cơ giới chạy với tốc độ vượt quá 70 km/h."], ["P.127", "Tốc độ tối đa 80 km/h", "Biển báo cấm", "Cấm các loại xe cơ giới chạy với tốc độ vượt quá 80 km/h."], ["DP.134", "Hết hạn chế tốc độ tối đa 80 km/h", "Biển hết lệnh cấm", "Hết đoạn đường hạn chế tốc độ tối đa 80 km/h."], ["P.127", "Tốc độ tối đa 100 km/h", "Biển báo cấm", "Cấm các loại xe cơ giới chạy với tốc độ vượt quá 100 km/h."], ["P.127", "Tốc độ tối đa 120 km/h", "Biển báo cấm", "Cấm các loại xe cơ giới chạy với tốc độ vượt quá 120 km/h."], ["P.125", "Cấm vượt", "Biển báo cấm", "Cấm các loại xe cơ giới vượt nhau trên đoạn đường đặt biển."], ["P.126", "Cấm xe ô tô tải vượt", "Biển báo cấm", "Cấm các loại xe ô tô tải vượt xe cơ giới khác."], ["W.207a", "Giao nhau với đường không ưu tiên", "Biển báo nguy hiểm", "Báo trước sắp đến nơi giao nhau với đường không ưu tiên; xe đi trên đường đang đi được quyền ưu tiên."], ["I.401", "Bắt đầu đường ưu tiên", "Biển chỉ dẫn", "Báo hiệu bắt đầu đoạn đường ưu tiên; xe đi trên đường này được ưu tiên khi qua nơi giao nhau."], ["W.208", "Giao nhau với đường ưu tiên (nhường đường)", "Biển báo nguy hiểm", "Báo trước sắp đến nơi giao nhau với đường ưu tiên; phải nhường đường cho xe đi trên đường ưu tiên."], ["R.122", "Dừng lại", "Biển hiệu lệnh", "Các xe phải dừng lại trước vạch dừng hoặc tại vị trí đặt biển; chỉ được đi tiếp khi bảo đảm an toàn."], ["P.101", "Đường cấm", "Biển báo cấm", "Cấm mọi loại phương tiện đi lại cả hai hướng, trừ các xe được ưu tiên theo quy định."], ["P.106a", "Cấm xe ô tô tải", "Biển báo cấm", "Cấm các loại xe ô tô tải đi vào đoạn đường đặt biển."], ["P.102", "Cấm đi ngược chiều", "Biển báo cấm", "Cấm các loại xe đi vào theo chiều đặt biển, trừ các xe được ưu tiên theo quy định."], ["W.233", "Nguy hiểm khác", "Biển báo nguy hiểm", "Báo phía trước có nguy hiểm mà không có biển cảnh báo riêng; cần chú ý quan sát, giảm tốc độ."], ["W.201a", "Chỗ ngoặt nguy hiểm vòng bên trái", "Biển báo nguy hiểm", "Báo trước sắp đến chỗ ngoặt nguy hiểm vòng sang trái; cần giảm tốc độ."], ["W.201b", "Chỗ ngoặt nguy hiểm vòng bên phải", "Biển báo nguy hiểm", "Báo trước sắp đến chỗ ngoặt nguy hiểm vòng sang phải; cần giảm tốc độ."], ["W.202a", "Nhiều chỗ ngoặt nguy hiểm liên tiếp", "Biển báo nguy hiểm", "Báo trước sắp đến đoạn đường có nhiều chỗ ngoặt nguy hiểm liên tiếp."], ["W.221a", "Đường có ổ gà, lồi lõm", "Biển báo nguy hiểm", "Báo trước đoạn đường mặt đường lồi lõm, ổ gà; cần giảm tốc độ."], ["W.222a", "Đường trơn", "Biển báo nguy hiểm", "Báo trước đoạn đường trơn trượt; cần giảm tốc độ, tránh phanh gấp."], ["W.203c", "Đường bị hẹp về phía phải", "Biển báo nguy hiểm", "Báo trước sắp đến đoạn đường bị hẹp đột ngột về phía phải."], ["W.227", "Công trường", "Biển báo nguy hiểm", "Báo trước sắp đến đoạn đường đang thi công; cần giảm tốc độ, chú ý người và máy làm việc."], ["W.209", "Giao nhau có tín hiệu đèn", "Biển báo nguy hiểm", "Báo trước sắp đến nơi giao nhau có điều khiển giao thông bằng đèn tín hiệu."], ["W.224", "Đường người đi bộ cắt ngang", "Biển báo nguy hiểm", "Báo trước sắp đến phần đường dành cho người đi bộ sang ngang; phải giảm tốc độ, nhường đường."], ["W.225", "Trẻ em", "Biển báo nguy hiểm", "Báo trước gần trường học, nơi trẻ em thường qua lại; phải giảm tốc độ, đặc biệt chú ý."], ["W.226", "Đường người đi xe đạp cắt ngang", "Biển báo nguy hiểm", "Báo trước nơi có người đi xe đạp cắt ngang đường; cần giảm tốc độ, chú ý."], ["—", "Đường có băng tuyết", "Không có trong QCVN", "Biển của bộ dữ liệu GTSRB (Đức); hệ thống biển báo Việt Nam không có biển tương ứng."], ["W.231", "Thú rừng vượt qua đường", "Biển báo nguy hiểm", "Báo trước nơi thường có thú rừng đi qua đường; cần giảm tốc độ, chú ý."], ["DP.135", "Hết tất cả các lệnh cấm", "Biển hết lệnh cấm", "Hết hiệu lực của tất cả các biển báo cấm đặt trước đó."], ["R.301*", "Rẽ phải phía trước", "Biển hiệu lệnh", "Các xe chỉ được rẽ phải ở phía trước."], ["R.301*", "Rẽ trái phía trước", "Biển hiệu lệnh", "Các xe chỉ được rẽ trái ở phía trước."], ["R.301a", "Đi thẳng", "Biển hiệu lệnh", "Các xe chỉ được đi thẳng."], ["R.301*", "Đi thẳng hoặc rẽ phải", "Biển hiệu lệnh", "Các xe chỉ được đi thẳng hoặc rẽ phải."], ["R.301*", "Đi thẳng hoặc rẽ trái", "Biển hiệu lệnh", "Các xe chỉ được đi thẳng hoặc rẽ trái."], ["R.302a", "Hướng phải đi vòng chướng ngại vật sang phải", "Biển hiệu lệnh", "Các xe phải đi vòng sang phía phải chướng ngại vật."], ["R.302b", "Hướng phải đi vòng chướng ngại vật sang trái", "Biển hiệu lệnh", "Các xe phải đi vòng sang phía trái chướng ngại vật."], ["R.303", "Nơi giao nhau chạy theo vòng xuyến", "Biển hiệu lệnh", "Các xe phải chạy vòng theo đảo an toàn ở nơi giao nhau, theo chiều mũi tên."], ["DP.133", "Hết cấm vượt", "Biển hết lệnh cấm", "Hết đoạn đường cấm vượt."], ["DP.133*", "Hết cấm xe ô tô tải vượt", "Biển hết lệnh cấm", "Hết đoạn đường cấm xe ô tô tải vượt."]]''')
Y_NGHIA = {'P.130': 'Cấm các loại xe dừng và đỗ ở phía đường có đặt biển.', 'P.131a': 'Cấm các loại xe đỗ ở phía đường có đặt biển.',
           'P.123a': 'Cấm các loại xe rẽ trái ở nơi giao nhau phía trước.', 'P.123b': 'Cấm các loại xe rẽ phải ở nơi giao nhau phía trước.',
           'P.124a*': 'Cấm các loại xe quay đầu.', 'P.124c': 'Cấm các loại xe rẽ trái và quay đầu.',
           'P.127*40': 'Cấm các loại xe cơ giới chạy với tốc độ vượt quá 40 km/h.', 'P.103a': 'Cấm các loại xe ô tô đi vào theo hướng đặt biển.',
           'P.104': 'Cấm xe mô tô, xe gắn máy đi vào theo hướng đặt biển.', 'P.117*': 'Cấm các xe có chiều cao (kể cả hàng) vượt quá trị số ghi trên biển.',
           'W.245a': 'Nhắc người lái xe giảm tốc độ, đi chậm.', 'W.210': 'Báo trước sắp đến chỗ giao nhau với đường sắt có rào chắn.'}
NHOM = {'P': 'Biển báo cấm', 'W': 'Biển báo nguy hiểm', 'I': 'Biển chỉ dẫn', 'R': 'Biển hiệu lệnh'}
signs = [{'id': i, 'code': c, 'name': n, 'group': g, 'meaning': m} for i, (c, n, g, m) in enumerate(SIGNS_43)]
os.makedirs('signs', exist_ok=True)
for k, t in enumerate(THEM):
    ten = TEN_VIE.get(t, t)
    if t == 'P.127*40':
        ten = 'Tốc độ tối đa 40 km/h'
    ma = t.split('*')[0]
    signs.append({'id': 43 + k, 'code': ma, 'name': ten, 'group': NHOM.get(t[0], 'Khác'), 'meaning': Y_NGHIA.get(t, ten + '.')})
    # ảnh minh họa: biển rõ nhất (lớn nhất) của loại đó
    ds = [h for h, y in zip(vn_hop, vn_y) if y == 43 + k]
    if not ds:
        continue
    anh, cx, cy, w, h = max(ds, key=lambda x: min(x[3], x[4]))
    Image.open(anh).convert('RGB').crop((cx - 0.55 * w, cy - 0.55 * h, cx + 0.55 * w, cy + 0.55 * h)).resize((256, 256)).save(f'signs/{43 + k}.png')
json.dump(signs, open('signs.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(pd.DataFrame(signs[43:])[['id', 'code', 'name']].to_string(index=False))

In [ ]:
# Bước 11: Thử nghiệm 100 biển báo Việt Nam (lấy ngẫu nhiên trong ảnh kiểm tra)
# Xử lý giống app: cắt khung vuông quanh biển, chạy 5 vùng cắt lệch nhau, đưa về 30x30,
# lấy trung bình xác suất, ngưỡng tin cậy 60%.
CAT = [(1.0, 0, 0), (0.9, 0, 0), (1.1, 0, 0), (1.0, 0.05, 0.05), (1.0, -0.05, -0.05)]

def cat_vuong(im, x0, y0, d):
    x0 = min(max(0, x0), im.width - 1)
    y0 = min(max(0, y0), im.height - 1)
    return im.crop((round(x0), round(y0), round(min(im.width, x0 + d)), round(min(im.height, y0 + d))))

chi_so = [i for i in range(len(vn_y)) if vn_phan[i] == 'test']
random.seed(42)
chon = random.sample(chi_so, min(100, len(chi_so)))
ket_qua = []
for k, i in enumerate(chon, 1):
    anh, cx, cy, w, h = vn_hop[i]
    im = Image.open(anh).convert('RGB')
    canh = max(w, h) * 1.2
    xs = []
    tg = 0
    for ti_le, dx, dy in CAT:
        d = canh * ti_le
        o = cat_vuong(im, cx + dx * canh - d / 2, cy + dy * canh - d / 2, d).resize((30, 30), Image.BICUBIC)
        t = time.perf_counter()
        it.set_tensor(VAO, np.array(o, np.float32)[None])
        it.invoke()
        xs.append(it.get_tensor(RA)[0])
        tg += (time.perf_counter() - t) * 1000
    p = np.mean(xs, axis=0)
    ket_qua.append({'Lần': k, 'Ảnh': os.path.basename(anh), 'Biển': vn_ten[i], 'Loại': 'mới' if vn_y[i] >= 43 else 'cũ',
                    'Nhãn đúng': int(vn_y[i]), 'Dự đoán': int(p.argmax()), 'Tên dự đoán': signs[int(p.argmax())]['name'],
                    'Độ tin cậy (%)': round(float(p.max()) * 100, 1), 'Vượt 60%': bool(p.max() >= 0.6),
                    'Đúng': int(p.argmax()) == int(vn_y[i]), 'Thời gian (ms)': round(tg, 2)})
kq = pd.DataFrame(ket_qua)
kq.to_csv('ket_qua_100_bien.csv', index=False, encoding='utf-8-sig')
for loai in ['cũ', 'mới']:
    d = kq[kq['Loại'] == loai]
    print(f'Biển {loai}: {len(d)} lần, đúng {d["Đúng"].sum()} ({d["Đúng"].mean() * 100:.1f}%)')
print(f'Tổng: đúng {kq["Đúng"].sum()}/100 | vượt 60%: {kq["Vượt 60%"].sum()} lần, trong đó đúng {kq[kq["Vượt 60%"]]["Đúng"].sum()}')
print(kq.groupby('Biển').agg(so_lan=('Đúng', 'size'), dung=('Đúng', 'sum')).sort_values('so_lan', ascending=False).to_string())

fig, axs = plt.subplots(10, 10, figsize=(14, 15))
for ax, i, (_, k) in zip(axs.flat, chon, kq.iterrows()):
    anh, cx, cy, w, h = vn_hop[i]
    canh = max(w, h) * 1.2
    ax.imshow(cat_vuong(Image.open(anh).convert('RGB'), cx - canh / 2, cy - canh / 2, canh).resize((96, 96)))
    mau_vien = 'green' if k['Đúng'] else 'red'
    ax.set_title(f"{k['Lần']} {k['Biển']} {k['Độ tin cậy (%)']:.0f}%", fontsize=7, color=mau_vien)
    ax.set_xticks([])
    ax.set_yticks([])
    for vien in ax.spines.values():
        vien.set_color(mau_vien)
        vien.set_linewidth(3)
plt.tight_layout()
plt.savefig('hinh_100_bien.png', dpi=150)
plt.show()

In [ ]:
# Bước 12: Số liệu cho báo cáo và tải file về
print('Số lớp:', SO_LOP, '(43 GTSRB +', len(THEM), 'biển Việt Nam mới) | tham số:', model.count_params())
print('Huấn luyện:', len(X_train), 'ảnh | kiểm định:', len(X_val), 'ảnh |', epochs, 'epoch | thời gian:', round(thoi_gian_huan_luyen, 1), 'phút')
print('Epoch cuối: huấn luyện', round(history.history['accuracy'][-1] * 100, 2), '% | kiểm định', round(history.history['val_accuracy'][-1] * 100, 2), '%')
print('GTSRB 8.000 ảnh:', int((y8 == p8).sum()), 'đúng |', round(accuracy_score(y8, p8) * 100, 2), '% | Macro P/R/F1:', round(P.mean(), 2), round(R.mean(), 2), round(F1.mean(), 2))
print('Biển Việt Nam (kiểm tra):', len(vn_te_y), 'biển |', round(accuracy_score(vn_te_y, pv) * 100, 2), '% | Macro F1:', round(Fv.mean(), 2))
print('100 lần thử: đúng', int(kq['Đúng'].sum()), '| thời gian chạy TB', round(kq['Thời gian (ms)'].mean(), 2), 'ms (Colab)')
print('model.tflite:', round(os.path.getsize('model.tflite') / 2**20, 2), 'MB')

with zipfile.ZipFile('ket_qua.zip', 'w') as z:
    for f in ['model.tflite', 'signs.json', 'traffic_classifier.h5', 'do_thi_huan_luyen.png', 'f1_gtsrb.csv',
              'f1_bien_viet_nam.csv', 'ket_qua_100_bien.csv', 'hinh_100_bien.png'] + sorted(glob.glob('signs/*.png')):
        z.write(f)
from google.colab import files
files.download('ket_qua.zip')